# Study 811 — Zero-Return Illiquidity 🕳️

**Do stocks that print *exactly-zero* daily returns earn an illiquidity premium?**

Lesmond, Ogden & Trzcinka (1999) argue that when the round-trip cost of trading
exceeds the day's information, the informed trader stays home and the price **doesn't
move** — so the *frequency of zero-return days* is a cheap, price-only proxy for a
name's transaction cost. Illiquid names should be compensated (Amihud & Mendelson
1986), so a long **high-zero** / short **low-zero** book should earn a positive spread.
We take the self-contained daily version on a liquid US cross-section (2010-01-04 →
2026-06-30, 50 names).

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. Survivorship: current-membership mega-caps — magnitudes are an upper
bound.*


## 1. The idea in one picture

A stock nobody trades sits still: with no trade, the close repeats and the day's return is **exactly zero**. Count how often that happens over the past year and you have a free illiquidity meter — the harder a name is to trade, the more zero-days it prints. Illiquidity should be *paid for* (Amihud & Mendelson), so buy the often-zero names, sell the never-zero ones.

## 2. The catch, before we even look at returns

This trick was built for tick-priced, thinly-traded small-caps. **Mega-caps almost never sit still.** On our 50-name universe the *median* stock prints a zero return on **0.00%** of trailing-year days, and the single most-frequent name only reaches **2.38%**. Half the list is pinned at exactly 0.00% — the sort has almost nothing to bite on. That alone tells us to expect **None** here.

In [1]:
import numpy as np, pandas as pd
R = dict(spread_bps=-1.37, t_nw=-1.29, hi_bps=6.55, lo_bps=7.93, gross_sharpe=-0.33,
         zp_med_pct=0.0, zp_max_pct=2.38)
print('trailing-year zero-return proportion: median %.2f%%  max %.2f%% of days'
      % (R['zp_med_pct'], R['zp_max_pct']))
print('long high-zero / short low-zero spread: %+.2f bps/day (NW t = %+.2f)'
      % (R['spread_bps'], R['t_nw']))
print('  illiquid book %+.2f bps vs liquid book %+.2f bps'
      % (R['hi_bps'], R['lo_bps']))
print('  gross spread Sharpe (before cost): %.2f' % R['gross_sharpe'])

trailing-year zero-return proportion: median 0.00%  max 2.38% of days
long high-zero / short low-zero spread: -1.37 bps/day (NW t = -1.29)
  illiquid book +6.55 bps vs liquid book +7.93 bps
  gross spread Sharpe (before cost): -0.33


## 3. Is the machinery even working? A live synthetic control

We plant the premium in a seeded toy world (`edge>0`, where often-zero names really do earn more) and check the detector recovers it — and that it stays *silent* on the null (`edge=0`, zero-days present but unpriced). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from zero_return import data, strategy as st
null = st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=811, n_assets=40, n_days=1200))
planted = st.synthetic_detect(data.synthetic_panel(edge=0.012, seed=811, n_assets=40, n_days=1500))
print('null world   : spread NW t = %+.2f  (should be ~0)' % null['t_nw'])
print('planted world: spread NW t = %+.2f  (should light up)' % planted['t_nw'])

null world   : spread NW t = +0.33  (should be ~0)
planted world: spread NW t = +11.23  (should light up)


## 4. The honest verdict — no premium here

On this liquid mega-cap tape the long-high-zero / short-low-zero spread is **-1.37 bps/day** with NW *t* = **-1.29** — statistically indistinguishable from zero (and if anything faintly the wrong way). A 1,000-permutation null centres at zero (sd 1.11 bps) and the observed value sits only ~1.4σ into the tail. The illiquidity premium is a small-and-illiquid-stock phenomenon; on 50 mega-caps the proxy is near-degenerate and there is nothing to harvest. **Signal: None**, **Tradability: Mirage** (the book loses money gross, and the long leg's real trading costs dwarf any edge).